# Deposit Attrition EDA — v9 · Where the money goes

**Payment Knowledge Graph (PKG) · PNC Treasury Management · Data Science**

v7 and v8b answered *who is leaving*. This notebook answers **how much money is leaving, where it
goes, how much of it is still on the books when we find out, and what an RM call is worth.**

## The reframe

Everything so far ranks by `P(exit)`. Every client counts the same, so a $5,000 client and a
$50,000,000 client sit side by side in the queue. The business does not care equally about them.

```
E[dollars saved | we alert on client c at month t]
      =  P(c exits within H)  ×  balance still on the books at t  ×  P(save | RM contacts them)
```

Three terms, and **we currently model only the first**. This notebook measures the second,
parameterises the third, and rebuilds the queue on the product.

## Two things that must be established before any savings number is credible

**1 · How much balance is left when we catch them.** If a client has already moved 80% of the money
by the time the model flags it, the alert is a post-mortem. §3 measures the decay curve in dollars
and it sets the ceiling on everything downstream.

**2 · Is the model as good on large clients as on small ones?** A value-weighted queue built on a
model that is weaker in the top balance decile is built on sand. §9 checks it, and if it fails, §7
and §8 are void.

## Blocks

| § | What | Why |
|---|---|---|
| 2 | **Balance at risk** — four candidate definitions, measured and compared | "The balance" is ambiguous and the choice changes the answer |
| 3 | **The decay curve** — dollars remaining at each month before exit | Sets the ceiling on what earlier warning is worth |
| 4 | **Concentration** — the Pareto of at-risk dollars | Decides whether a short value-ranked list can cover most of the money |
| 5 | **Where the money goes** — balance decline reconciled against payment outflow, destination institutions, self-directed transfers | The literal question, and the part most likely to be over-read |
| 6 | **Balance as a feature** — level, volatility, band. Never used; only the dd has been | Cheap ablation block |
| 7 | **Value-weighted ranking** — rank on `P × balance^α` | The deliverable |
| 8 | **The savings model** — grid over alert volume × RM success rate, decay-adjusted, with break-even | What the manager reads |
| 9 | **Validation** — calibration, and precision by balance decile | Without these, §7 and §8 are not defensible |

## What this notebook will not claim

There is **no data on RM save rates**. Section 8 is a sensitivity grid, not a forecast. Every
savings figure is conditional on an assumption that only a pilot can settle, and the notebook says
so in its own output.

## 0 · Configuration

In [ ]:
# =====================================================================
# 0 · CONFIGURATION — v9
# =====================================================================
from pathlib import Path

HDFS_V2  = "hdfs://nameservice1/user/pk36814/attrition_v2"
HDFS_V6  = "hdfs://nameservice1/user/pk36814/attrition_v6"
HDFS_V7  = "hdfs://nameservice1/user/pk36814/attrition_v7"
HDFS_V8  = "hdfs://nameservice1/user/pk36814/attrition_v8"
HDFS_DIR = "hdfs://nameservice1/user/pk36814/attrition_v9"
OUT_DIR  = Path("/projects/DSI/sa15474/repos/pkg/eda/attrition_v9")
# TRAP: pathlib collapses hdfs://host/p -> hdfs:/host/p. Local Path and HDFS
# string stay separate variables and are never mixed.

DATE_START, DATE_END = "2024-01-01", "2026-07-31"
MAX_ROWS, SEED = 60, 20260908

# ── carried unchanged so everything stays comparable to v8b ───────────
CHG_LAG_FAR, CHG_LAG_NEAR, CHG_MIN_OBS, MIN_REF = -6, -4, 2, 1.0
PEER_MIN_N, DD_CLIP = 50, (0.01, 100.0)
DD_WARMUP_M = 6                 # OFFSET. m_idx is ABSOLUTE (~24289-24319)
ORIGIN_START_OFF, MAX_ORIGINS = 18, 24
HORIZONS, PRIMARY_H, PRIMARY_DEF = [1, 3, 6], 6, "A_full_exit"
NEG_SAMPLE, MIN_TRAIN_POS = 0.15, 300
L2, IRLS_MAX_IT, IRLS_TOL = 2.0, 60, 1e-9
MAX_COLLECT_ROWS = 3_500_000
COOLDOWN_M = 3

# ── §2 balance at risk ────────────────────────────────────────────────
BAR_WINDOW  = 12          # trailing months for the robust balance
BAR_DEF     = "med12"     # decided empirically in §2 — one of
                          # {now, med12, peak12, mean12}
BAL_FLOOR   = 1_000.0     # below this a client is not worth an RM call
DECILE_N    = 10

# ── §3 decay ──────────────────────────────────────────────────────────
DECAY_PRE, DECAY_POST = 18, 3

# ── §5 where the money goes ───────────────────────────────────────────
DEST_WINDOW    = (-6, 0)  # rel_m window for destination analysis
DEST_BASE      = (-18, -12)
DEST_MIN_AMT   = 10_000.0
DEST_TOP_N     = 40
SELF_SIM_MIN   = 0.90

# ── §7/§8 the value queue and the savings model ───────────────────────
CAPACITY     = [50, 100, 250, 500, 1000, 2500, 5000]
QUEUE_K      = 250
VALUE_ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]   # score = p * bar**alpha
                                             # alpha=0 is today's queue
P_SAVE_GRID  = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40]
P_SAVE_BASE  = 0.20       # ASSUMPTION — no data. Sensitivity, not a forecast
RM_COST_PER_CALL = 250.0  # loaded cost of one RM outreach. Override freely
ANNUALISE    = 12.0/7.0   # the test window is 7 months

# ── the shipping specification from v8b ───────────────────────────────
WIN_BLOCKS = ["fin_in", "cpty_acct_out"]
HTML_NAME  = "PKG_Attrition_v9_Money.html"

RUN_DECAY, RUN_DEST, RUN_FEATURES, RUN_QUEUE, RUN_SAVINGS = True, True, True, True, True


In [ ]:
# =====================================================================
# 1 · IMPORTS, SESSION, HELPERS
# =====================================================================
import warnings, time, math, json, datetime as dt
import numpy as np, pandas as pd
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark import StorageLevel
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)
spark = (SparkSession.builder.appName("pkg_attrition_eda_v9")
         .config("spark.sql.shuffle.partitions", "800")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         # KEEP OFF. PySpark 3.3.2's arrow path references np.object0/np.bool8,
         # both removed in numpy 2.0.
         .config("spark.sql.autoBroadcastJoinThreshold", str(64*1024*1024))
         .enableHiveSupport().getOrCreate())
pd.set_option("display.max_columns", 400); pd.set_option("display.width", 260)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"numpy {np.__version__} · pandas {pd.__version__} · spark {spark.version}")

def hp(n): return f"{HDFS_DIR.rstrip('/')}/{n}"
def v2(n): return f"{HDFS_V2.rstrip('/')}/{n}"
def v6(n): return f"{HDFS_V6.rstrip('/')}/{n}"
def v7(n): return f"{HDFS_V7.rstrip('/')}/{n}"
def v8(n): return f"{HDFS_V8.rstrip('/')}/{n}"
def exists(p):
    try: spark.read.parquet(p).limit(1).count(); return True
    except Exception: return False
def pct(a, b): return float(a)/float(b) if b else float("nan")

def _dec(s):
    o = s
    for c, t in s.dtypes:
        if t.startswith("decimal"): o = o.withColumn(c, F.col(c).cast("double"))
    return o

def disp(o, title=None, n=None, save=None):
    n = MAX_ROWS if n is None else n
    out = _dec(o).limit(n).toPandas() if hasattr(o, "toPandas") else (
        o.copy() if isinstance(o, pd.DataFrame) else pd.DataFrame(o))
    if save: out.to_csv(OUT_DIR / f"{save}.csv", index=False)
    if title:
        display(HTML(f"<div style='font:600 13px/1.6 IBM Plex Sans,sans-serif;"
                     f"margin:10px 0 2px;color:#111'>{title}"
                     f"<span style='font-weight:400;color:#888'> &middot; {len(out)} rows"
                     f"</span></div>"))
    display(out); return out

def kv(pairs, title=None, save=None):
    """Ordered PAIRS, raises on a repeated label — the v6 duplicate-key bug
    cannot recur."""
    items = list(pairs.items()) if isinstance(pairs, dict) else list(pairs)
    labs = [k for k, _ in items]
    d = sorted({k for k in labs if labs.count(k) > 1})
    if d: raise ValueError(f"kv(): duplicate labels {d}")
    return disp(pd.DataFrame({"metric": labs, "value": [v for _, v in items]}),
                title=title, n=len(items), save=save)

def collect_pd(sdf, label="", max_rows=None):
    max_rows = MAX_COLLECT_ROWS if max_rows is None else max_rows
    n = sdf.count()
    if n > max_rows: raise RuntimeError(f"{label}: {n:,} > {max_rows:,}")
    t0 = time.time(); out = _dec(sdf).toPandas()
    print(f"  collected {label}: {n:,} x {out.shape[1]} in {time.time()-t0:,.0f}s")
    return out

# ── money formatting: everything in this notebook is dollars ──────────
def usd(v):
    if v is None or (isinstance(v, float) and not np.isfinite(v)): return "—"
    a = abs(v)
    if a >= 1e9:  return f"${v/1e9:,.2f}bn"
    if a >= 1e6:  return f"${v/1e6:,.1f}m"
    if a >= 1e3:  return f"${v/1e3:,.0f}k"
    return f"${v:,.0f}"
def usd_col(df, cols):
    d = df.copy()
    for c in ([cols] if isinstance(cols, str) else cols):
        if c in d.columns: d[c] = d[c].map(usd)
    return d

# ── metrics carried from v8b ──────────────────────────────────────────
def auc(y, s):
    y = np.asarray(y, float); s = np.asarray(s, float)
    ok = np.isfinite(s) & np.isfinite(y); y, s = y[ok], s[ok]
    n1 = float(y.sum()); n0 = float(len(y)-n1)
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(s).rank(method="average").to_numpy()
    return float((r[y == 1].sum() - n1*(n1+1)/2.0)/(n1*n0))

def logit_irls(X, y, l2=L2, mi=IRLS_MAX_IT, tol=IRLS_TOL):
    X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
    b = np.zeros(X.shape[1]); R = l2*np.eye(X.shape[1]); R[0, 0] = 0.0
    for _ in range(mi):
        eta = np.clip(X @ b, -30, 30); mu = 1/(1+np.exp(-eta))
        w = np.maximum(mu*(1-mu), 1e-6); z = eta + (y-mu)/w; XtW = X.T*w
        try: bn = np.linalg.solve(XtW @ X + R, XtW @ z)
        except np.linalg.LinAlgError:
            bn = np.linalg.lstsq(XtW @ X + R, XtW @ z, rcond=None)[0]
        if np.max(np.abs(bn-b)) < tol: b = bn; break
        b = bn
    return b

def fit_spec(tr, cols, l2=L2, s=NEG_SAMPLE):
    """King & Zeng prior correction on the intercept. Ranking is unaffected;
    the CALIBRATED PROBABILITY is not — and v9 multiplies it by dollars, so it
    has to be right. §9 checks it."""
    X = tr[cols].to_numpy(np.float64); y = tr["y"].to_numpy(np.float64)
    keep = X.std(axis=0) > 1e-9
    ck = [c for c, k in zip(cols, keep) if k]
    if not ck: return None
    Xk = X[:, keep]; mu = Xk.mean(0); sd = Xk.std(0)
    b = logit_irls(np.column_stack([np.ones(len(Xk)), (Xk-mu)/sd]), y, l2)
    return dict(cols=ck, beta=b[1:]/sd,
                b0=float(b[0]-float(np.sum(b[1:]*mu/sd))) + math.log(s))

def predict_p(sp, df):
    """Returns a PROBABILITY, not a score. v9 needs the scale."""
    if sp is None: return np.full(len(df), np.nan)
    eta = sp["b0"] + df[sp["cols"]].to_numpy(np.float64) @ sp["beta"]
    return 1.0/(1.0 + np.exp(-np.clip(eta, -30, 30)))

_F = OUT_DIR / "FINDINGS_v9.csv"
FINDINGS = pd.read_csv(_F).to_dict("records") if _F.exists() else []
def note(qid, q, a, d=""):
    global FINDINGS
    FINDINGS = [f for f in FINDINGS if f["id"] != qid]
    FINDINGS.append(dict(id=qid, question=q, answer=str(a), detail=str(d)))
    pd.DataFrame(FINDINGS).to_csv(_F, index=False)
print("helpers ready")


## 2 · Balance at risk — pick a definition before using one

"The client's balance" is ambiguous and the choice changes every number downstream. Four
candidates, all measured at the scoring month *t*:

| Candidate | Definition | Bias |
|---|---|---|
| `bal_now` | `bal_live` at *t* | What is actually there to defend. **Understates** for a client already draining |
| `bal_med12` | median `bal_live` over *t−11 … t* | Robust to a single spike. **Overstates** what is still recoverable |
| `bal_mean12` | mean over the same window | Sensitive to one large month |
| `bal_peak12` | max over the same window | The most flattering, and the least defensible |

Two of these answer different questions and both are needed. **`bal_now` is what an RM can still
save.** **`bal_med12` is the size of the relationship** and is the right thing to rank on, because
a temporarily-drained large client is still worth a call.

The gap between them, summed over attriters, is the money that has already gone by the time we
alert. That number is §3.

In [ ]:
# =====================================================================
# 2 · BALANCE AT RISK                                    [OUTPUT BLOCK 1]
# =====================================================================
t0 = time.time()
cust_month = spark.read.parquet(v2("panel_customer_month")).filter(F.col("ym") >= DATE_START[:7])
YMMAP = cust_month.select("ym", "m_idx").distinct().persist(StorageLevel.DISK_ONLY)
M_MIN, M_MAX = [int(x) for x in cust_month.agg(F.min("m_idx"), F.max("m_idx")).collect()[0]]
print(f"  m_idx {M_MIN}-{M_MAX} (ABSOLUTE index — every month constant is an offset)")

lab = spark.read.parquet(v6("labels_customer")).persist(StorageLevel.DISK_ONLY)

wb = Window.partitionBy("cust_pwr_id").orderBy("m_idx").rangeBetween(-(BAR_WINDOW-1), 0)
BAL = (cust_month.select("cust_pwr_id", "m_idx", "ym", "bal_live", "n_accts", "n_accts_live")
       .withColumn("bal_now", F.greatest(F.coalesce("bal_live", F.lit(0.0)), F.lit(0.0)))
       .withColumn("_a", F.array_sort(F.collect_list("bal_now").over(wb)))
       .withColumn("bal_med12", F.expr("element_at(_a, cast(size(_a)/2 as int) + 1)")).drop("_a")
       .withColumn("bal_mean12", F.avg("bal_now").over(wb))
       .withColumn("bal_peak12", F.max("bal_now").over(wb))
       .withColumn("bal_n12", F.count("bal_now").over(wb))
       # volatility of the balance — never used as a feature, added in §6
       .withColumn("bal_sd12", F.stddev("bal_now").over(wb))
       .withColumn("bal_cv12", F.when(F.col("bal_mean12") > 0,
                                      F.col("bal_sd12")/F.col("bal_mean12")))
      ).persist(StorageLevel.DISK_ONLY)
BAL.write.mode("overwrite").partitionBy("m_idx").parquet(hp("balance"))
BAL = spark.read.parquet(hp("balance")).persist(StorageLevel.DISK_ONLY)

cands = ["bal_now", "bal_med12", "bal_mean12", "bal_peak12"]
S = (BAL.filter(F.col("bal_n12") >= 6).agg(
        F.count(F.lit(1)).alias("client_months"),
        *[F.sum(c).alias(f"total_{c}") for c in cands],
        *[F.expr(f"percentile_approx({c}, 0.5)").alias(f"p50_{c}") for c in cands],
        *[F.expr(f"percentile_approx({c}, 0.99)").alias(f"p99_{c}") for c in cands])
     ).toPandas().T
S.columns = ["value"]
disp(usd_col(S.reset_index().rename(columns={"index": "metric"}), []),
     title="2a &middot; The four candidates, over all client-months", n=20,
     save="v9_bar_candidates")

# what the whole book is worth, and what walks out
att = lab.filter(F.col("q_A_full_exit").isNotNull()).select(
        "cust_pwr_id", F.col("q_A_full_exit").alias("event_m"))
BOOK = BAL.filter(F.col("m_idx") == M_MAX).agg(F.sum("bal_now").alias("book")).collect()[0][0]
# the pool at risk: each attriter's relationship size, measured a year before it leaves
POOL = (BAL.join(att, "cust_pwr_id", "inner")
        .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
        .filter(F.col("rel_m") == -12)
        .agg(F.count(F.lit(1)).alias("n"), F.sum("bal_med12").alias("pool"),
             F.expr("percentile_approx(bal_med12, 0.5)").alias("median")).collect()[0])
kv([("live book at panel end", usd(BOOK)),
    ("attriters with a reading at rel_m -12", f"{POOL['n']:,}"),
    ("their relationship size a year out (sum of bal_med12)", usd(POOL["pool"])),
    ("median attriter relationship size", usd(POOL["median"])),
    ("that pool as a share of the live book", f"{pct(POOL['pool'], BOOK):.1%}"),
    ("annualised (x 12/31 months of events)", usd(POOL["pool"]*12/31)),
    ("block 2 wall (s)", round(time.time()-t0))],
   title="2b &middot; <b>The addressable pool.</b> This is the money attached to clients who "
         "later leave, measured a year before they leave — before any decline has started",
   save="v9_pool")
note("POOL", "How much money is attached to clients who leave?",
     usd(POOL["pool"]),
     "Sum of bal_med12 at rel_m -12 over qualified A_full_exit attriters. Measured a year out, "
     "so it is the size of the relationship, not what is left at the end. §3 measures how much "
     "of it survives to the month we alert.")


## 3 · The decay curve — how much is left when we find them

This is the ceiling on everything in §8. A queue that catches a client two months before it exits
can only defend whatever balance is still on the books at that point.

Measured in event time on purpose: index each attriter's balance to its own level at `rel_m −12`
and trace it forward. The stayer line is the control — some of the decline is seasonal or
market-wide and only the gap is attrition.

In [ ]:
# =====================================================================
# 3 · THE DECAY CURVE                                    [OUTPUT BLOCK 2]
# =====================================================================
if RUN_DECAY:
    t0 = time.time()
    # cohorts: attriters, plus stayers given pseudo-events from the same months
    ev = (lab.filter(F.col("q_A_full_exit").isNotNull())
          .select("cust_pwr_id", F.col("q_A_full_exit").alias("event_m"),
                  F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
          .filter(F.col("event_m") - F.col("first_m") >= DECAY_PRE)
          .withColumn("cohort", F.lit("attriter")))
    draw = [r.event_m for r in ev.select("event_m").limit(3000).collect()]
    draw = draw[::max(1, len(draw)//300)][:300] or [M_MIN + DECAY_PRE]
    arr = F.array(*[F.lit(int(x)) for x in draw])
    st = (lab.filter(F.col("q_A_full_exit").isNull() & F.col("q_B_bal_exit").isNull())
          .withColumn("event_m", F.element_at(arr,
                (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id", F.lit(SEED)))) % len(draw)) + 1))
          .select("cust_pwr_id", "event_m",
                  F.coalesce("first_live_m", "first_m").alias("first_m"), "last_m")
          .filter(F.col("event_m") - F.col("first_m") >= DECAY_PRE)
          .withColumn("cohort", F.lit("stayer")))
    CO = ev.unionByName(st).filter(F.col("last_m") >= F.col("event_m")) \
           .persist(StorageLevel.DISK_ONLY)

    ES = (BAL.join(CO, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
          .filter(F.col("rel_m").between(-DECAY_PRE, DECAY_POST)))
    # index each client to its OWN level at rel_m -12
    anchor = (ES.filter(F.col("rel_m") == -12)
              .select("cust_pwr_id", F.col("bal_med12").alias("bal_anchor"))
              .filter(F.col("bal_anchor") > BAL_FLOOR))
    ES = ES.join(anchor, "cust_pwr_id", "inner") \
           .withColumn("idx", F.col("bal_now")/F.col("bal_anchor"))

    DEC = (ES.groupBy("cohort", "rel_m").agg(
              F.count(F.lit(1)).alias("n"),
              F.expr("percentile_approx(idx, 0.5)").alias("median_idx"),
              (F.sum("bal_now")/F.sum("bal_anchor")).alias("dollar_weighted_idx"),
              F.sum("bal_now").alias("dollars_remaining"),
              F.sum("bal_anchor").alias("dollars_anchor"))
           .orderBy("cohort", "rel_m")).toPandas()
    DEC.to_csv(OUT_DIR / "v9_decay.csv", index=False)
    piv = DEC.pivot_table(index="rel_m", columns="cohort",
                          values="dollar_weighted_idx").reset_index()
    piv["gap"] = (piv.get("stayer") - piv.get("attriter")).round(4)
    disp(piv.round(4), title="3a &middot; <b>Dollar-weighted balance index</b>, each client "
         "against its own level a year before the event. The stayer line is the control — only "
         "the gap is attrition", n=30, save="v9_decay_index")

    A = DEC[DEC.cohort == "attriter"].set_index("rel_m")
    def rem(rm): return float(A.loc[rm, "dollar_weighted_idx"]) if rm in A.index else np.nan
    rows = []
    for rm in [-12, -9, -6, -4, -3, -2, -1, 0]:
        r = rem(rm)
        rows.append(dict(alert_at_rel_m=rm, share_of_relationship_still_on_book=r,
                         already_gone=1-r if np.isfinite(r) else np.nan,
                         defendable_dollars=POOL["pool"]*r if np.isfinite(r) else np.nan))
    DEF = pd.DataFrame(rows)
    disp(usd_col(DEF, "defendable_dollars"),
         title="3b &middot; <b>The ceiling on any savings claim.</b> Alerting at rel_m &minus;2 "
               "can only defend what is left at rel_m &minus;2", save="v9_defendable")

    _2, _4 = rem(-2), rem(-4)
    kv([("relationship dollars still on book at rel_m -12", f"{rem(-12):.3f}"),
        ("...at -6", f"{rem(-6):.3f}"), ("...at -4", f"{_4:.3f}"),
        ("...at -2", f"{_2:.3f}"), ("...at -1", f"{rem(-1):.3f}"),
        ("value of moving the alert from -2 to -4",
         usd(POOL["pool"]*(_4-_2)) if np.isfinite(_2) and np.isfinite(_4) else "—"),
        ("block 3 wall (s)", round(time.time()-t0))],
       title="3c &middot; Reading 3a. <b>This converts lead time into dollars</b> — the number "
             "that makes a 2-month vs 4-month median lead a commercial argument rather than a "
             "statistic", save="v9_decay_headline")
    note("DECAY", "How much money is left when we alert?",
         f"{_2:.3f} of the relationship at rel_m -2, {_4:.3f} at -4",
         "Dollar-weighted, indexed to each client's own level at rel_m -12, controlled against "
         "stayers. This is the ceiling on every savings figure in §8.")


## 4 · Concentration — can a short list cover most of the money?

If at-risk dollars are as concentrated as payment volume is in the graph (top 0.1% of nodes hold
97.3% of dollars), then a 250-name list ranked by value could cover a large share of the money
while a 250-name list ranked by probability covers almost none of it. That is the entire case for
§7, so it gets measured rather than assumed.

In [ ]:
# =====================================================================
# 4 · CONCENTRATION OF AT-RISK DOLLARS                   [OUTPUT BLOCK 3]
# =====================================================================
AT = (BAL.join(att, "cust_pwr_id", "inner")
      .withColumn("rel_m", F.col("m_idx") - F.col("event_m"))
      .filter(F.col("rel_m") == -12)
      .select("cust_pwr_id", F.col("bal_med12").alias("bar")))
P = collect_pd(AT.filter(F.col("bar") > 0), "attriter balances")
P = P.sort_values("bar", ascending=False).reset_index(drop=True)
P["cum_share"] = P.bar.cumsum()/P.bar.sum()
P["client_share"] = (P.index+1)/len(P)
rows = []
for q in [0.001, 0.005, 0.01, 0.02, 0.05, 0.10, 0.25, 0.50]:
    k = max(1, int(round(q*len(P))))
    rows.append(dict(top_share_of_clients=q, n_clients=k,
                     dollars=float(P.bar.head(k).sum()),
                     share_of_at_risk_dollars=float(P.cum_share.iloc[k-1])))
LOR = pd.DataFrame(rows)
disp(usd_col(LOR, "dollars"),
     title="4a &middot; <b>Concentration of at-risk dollars.</b> If the top 1% of departing "
           "clients hold a large share, a value-ranked list of a few hundred names covers most "
           "of the money", save="v9_concentration")

srt = np.sort(P.bar.values)
lorenz = srt.cumsum()/srt.sum()
gini = float(1 - 2*np.trapezoid(lorenz, dx=1/len(srt)))   # numpy 2: trapezoid, not trapz
kv([("departing clients with a balance", f"{len(P):,}"),
    ("total at-risk dollars", usd(P.bar.sum())),
    ("median", usd(P.bar.median())), ("mean", usd(P.bar.mean())),
    ("p90", usd(P.bar.quantile(.90))), ("p99", usd(P.bar.quantile(.99))),
    ("max", usd(P.bar.max())),
    ("Gini of at-risk dollars", round(gini, 4)),
    ("share of dollars in the top 1% of departing clients",
     f"{float(LOR.loc[LOR.top_share_of_clients==0.01,'share_of_at_risk_dollars'].iloc[0]):.1%}")],
   title="4b &middot; The shape of the pool", save="v9_concentration_headline")

# balance deciles of the AT-RISK book, used everywhere downstream
DEC_EDGES = list(np.quantile(P.bar.values, np.linspace(0, 1, DECILE_N+1)))
kv([(f"decile {i+1} upper bound", usd(DEC_EDGES[i+1])) for i in range(DECILE_N)],
   title="4c &middot; Balance decile boundaries on departing clients — carried into §9",
   save="v9_decile_edges")
note("CONC", "Are at-risk dollars concentrated enough for a value-ranked list to matter?",
     f"Gini {gini:.3f}; top 1% hold "
     f"{float(LOR.loc[LOR.top_share_of_clients==0.01,'share_of_at_risk_dollars'].iloc[0]):.1%}",
     "If concentration is high, ranking on probability alone leaves most of the money out of the "
     "queue. §7 tests whether ranking on P x balance recovers it.")


## 5 · Where the money goes

Three separate questions, in increasing order of how much they can actually tell you. **They are
kept apart deliberately, because the first is the one most likely to be over-read.**

**5a · Is the money even visible as payments?** A balance can fall because the client wired it out,
or because inflows stopped while outflows continued, or because of a single lump-sum movement we
cannot see. Before attributing destinations, reconcile the balance decline against observable
outbound payments. If the decline is much larger than the outflow, "where the money goes" is
partly unanswerable from this data and the rest of this section is bounded by that.

**5b · Which institutions receive it.** Now measurable: `cpty_fin_entity_name` covers **96% of
outbound dollars**. But **a payment to Chase does not mean the client moved to Chase** — it means
they paid someone who banks at Chase. Suppliers, payroll processors and card networks dominate.
This is read as a *relative* signal: institutions that receive disproportionately from
soon-to-exit clients, controlled against stayers.

**5c · Self-directed transfers.** A payment to a counterparty carrying the **client's own name** at
a **named non-PNC institution**. This is the only construct here that is unambiguous: it is the
client moving its own money to its own account somewhere else. v8 built it as a `dd` and it died
at 8.4% coverage; here it is read as a **dollar level and share**, which is the right scale for a
sparse, high-value event.

In [ ]:
# =====================================================================
# 5a · DOES THE BALANCE DECLINE SHOW UP AS PAYMENTS?     [OUTPUT BLOCK 4]
# =====================================================================
if RUN_DEST:
    t0 = time.time()
    feat = spark.read.parquet(v2("panel_pay_features")).filter(F.col("ym") >= DATE_START[:7])
    FL = (BAL.select("cust_pwr_id", "m_idx", "bal_now")
          .join(feat.select("cust_pwr_id", "m_idx",
                            F.coalesce("amt_out", F.lit(0.0)).alias("amt_out"),
                            F.coalesce("amt_in", F.lit(0.0)).alias("amt_in")),
                ["cust_pwr_id", "m_idx"], "left")
          .withColumn("amt_out", F.coalesce("amt_out", F.lit(0.0)))
          .withColumn("amt_in", F.coalesce("amt_in", F.lit(0.0))))
    w1 = Window.partitionBy("cust_pwr_id").orderBy("m_idx")
    FL = (FL.withColumn("bal_prev", F.lag("bal_now").over(w1))
            .withColumn("d_bal", F.col("bal_now") - F.col("bal_prev"))
            .withColumn("net_pay", F.col("amt_in") - F.col("amt_out"))
            .withColumn("unexplained", F.col("d_bal") - F.col("net_pay")))
    REC = (FL.join(att, "cust_pwr_id", "left")
           .withColumn("cohort", F.when(F.col("event_m").isNotNull(), "attriter")
                                  .otherwise("stayer"))
           .withColumn("rel_m", F.when(F.col("event_m").isNotNull(),
                                       F.col("m_idx")-F.col("event_m")))
           .filter(F.col("bal_prev").isNotNull()))
    A = (REC.filter((F.col("cohort") == "attriter") & F.col("rel_m").between(-12, 0))
         .groupBy("rel_m").agg(
             F.count(F.lit(1)).alias("n"),
             F.sum("d_bal").alias("balance_change"),
             F.sum("net_pay").alias("net_payment_flow"),
             F.sum("amt_out").alias("outbound"),
             F.sum("amt_in").alias("inbound"),
             F.sum("unexplained").alias("unexplained"))
         .orderBy("rel_m")).toPandas()
    A["explained_share"] = (A.net_payment_flow/A.balance_change).replace([np.inf, -np.inf], np.nan)
    disp(usd_col(A, ["balance_change", "net_payment_flow", "outbound", "inbound", "unexplained"]),
         title="5a &middot; <b>Balance decline reconciled against observable payment flow</b>, "
               "attriters by month before exit. <code>explained_share</code> near 1.0 means the "
               "money left through payments we can see", n=20, save="v9_reconciliation")
    _tot = A[A.rel_m.between(-6, 0)]
    kv([("balance change, rel_m -6..0", usd(_tot.balance_change.sum())),
        ("net payment flow over the same window", usd(_tot.net_payment_flow.sum())),
        ("share of the decline explained by payments",
         f"{pct(_tot.net_payment_flow.sum(), _tot.balance_change.sum()):.1%}"),
        ("unexplained", usd(_tot.unexplained.sum())),
        ("interpretation", "high explained share -> destinations in 5b are meaningful; "
                           "low -> most of the money leaves invisibly")],
       title="5a2 &middot; <b>The bound on everything in 5b and 5c</b>",
       save="v9_reconciliation_headline")
    note("RECON", "Does the balance decline show up as payments we can see?",
         f"{pct(_tot.net_payment_flow.sum(), _tot.balance_change.sum()):.1%} explained over "
         f"rel_m -6..0",
         "Anything not explained left by a route the payment table does not capture. This is the "
         "honest bound on destination analysis.")


In [ ]:
# =====================================================================
# 5b · WHICH INSTITUTIONS RECEIVE THE MONEY               [OUTPUT BLOCK 5]
# =====================================================================
# Outbound dollars by receiving institution, from the v8 pair table
# (kt='fin', dir='out'). Institution coverage is 37% of TRANSACTIONS but
# 96% of DOLLARS, which is why this is answerable at all.
if RUN_DEST:
    PAIRS = (spark.read.option("basePath", v8("pairs")).parquet(v8("pairs"))
             .join(YMMAP, "ym", "inner"))
    FINOUT = PAIRS.filter((F.col("kt") == "fin") & (F.col("dir") == "out")) \
                  .select("cust_pwr_id", "m_idx", F.col("k").alias("fin"), "amt", "n_txn")

    ATT_W = (FINOUT.join(att, "cust_pwr_id", "inner")
             .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
             .filter(F.col("rel_m").between(*DEST_WINDOW)))
    n_att = att.count()
    STY = lab.filter(F.col("q_A_full_exit").isNull()).select("cust_pwr_id")
    n_sty = STY.count()
    STY_W = FINOUT.join(F.broadcast(STY) if n_sty < 200000 else STY, "cust_pwr_id", "inner") \
                  .filter(F.col("m_idx").between(M_MAX-6, M_MAX))

    a = (ATT_W.groupBy("fin").agg(F.sum("amt").alias("att_amt"),
                                  F.countDistinct("cust_pwr_id").alias("att_clients")))
    s = (STY_W.groupBy("fin").agg(F.sum("amt").alias("sty_amt"),
                                  F.countDistinct("cust_pwr_id").alias("sty_clients")))
    D = (a.join(s, "fin", "outer")
         .fillna({"att_amt": 0.0, "sty_amt": 0.0, "att_clients": 0, "sty_clients": 0})
         .withColumn("att_per_client", F.col("att_amt")/F.lit(max(n_att, 1)))
         .withColumn("sty_per_client", F.col("sty_amt")/F.lit(max(n_sty, 1)))
         # the discriminative quantity: dollars per departing client relative to
         # dollars per staying client. >1 means over-represented before an exit.
         .withColumn("over_index", F.when(F.col("sty_per_client") > 0,
                     F.col("att_per_client")/F.col("sty_per_client")))
         .filter(F.col("att_amt") >= DEST_MIN_AMT))
    TOP = collect_pd(D.orderBy(F.desc("att_amt")).limit(DEST_TOP_N), "top destinations by dollars")
    disp(usd_col(TOP[["fin", "att_amt", "att_clients", "sty_per_client", "att_per_client",
                      "over_index"]], ["att_amt", "sty_per_client", "att_per_client"]),
         title="5b1 &middot; <b>Where outbound dollars go in the six months before an exit</b>, "
               "by receiving institution. <b>Read this as flow, not as defection</b> — a payment "
               "to a bank usually means paying a supplier who banks there",
         n=DEST_TOP_N, save="v9_destinations_by_dollars")

    OVR = collect_pd(D.filter((F.col("att_clients") >= 25) & F.col("over_index").isNotNull())
                     .orderBy(F.desc("over_index")).limit(DEST_TOP_N),
                     "over-indexed destinations")
    disp(usd_col(OVR[["fin", "over_index", "att_clients", "att_amt", "att_per_client",
                      "sty_per_client"]], ["att_amt", "att_per_client", "sty_per_client"]),
         title="5b2 &middot; <b>Institutions that receive disproportionately from clients about "
               "to leave.</b> Controlled per-client against stayers. A lead worth investigating, "
               "not evidence of defection", n=DEST_TOP_N, save="v9_destinations_overindexed")

    # new destinations: money to an institution the client did not pay in the baseline
    BASE_FIN = (FINOUT.join(att, "cust_pwr_id", "inner")
                .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
                .filter(F.col("rel_m").between(*DEST_BASE))
                .select("cust_pwr_id", "fin").distinct()
                .withColumn("in_base", F.lit(1)))
    NEWD = (ATT_W.join(BASE_FIN, ["cust_pwr_id", "fin"], "left")
            .withColumn("is_new", F.col("in_base").isNull().cast("int")))
    NEWS = (NEWD.groupBy("rel_m").agg(
                F.sum(F.when(F.col("is_new") == 1, F.col("amt")).otherwise(0.0)).alias("new_fin_amt"),
                F.sum("amt").alias("total_amt"),
                F.countDistinct(F.when(F.col("is_new") == 1, F.col("fin"))).alias("n_new_fin"))
            .withColumn("share_to_new", F.col("new_fin_amt")/F.col("total_amt"))
            .orderBy("rel_m")).toPandas()
    disp(usd_col(NEWS, ["new_fin_amt", "total_amt"]),
         title="5b3 &middot; <b>Dollars going to institutions the client did not pay a year "
               "earlier.</b> A rising share is money finding a new home",
         save="v9_new_destinations")


In [ ]:
# =====================================================================
# 5c · SELF-DIRECTED TRANSFERS — the unambiguous construct
# =====================================================================
# Outbound to a counterparty carrying the CLIENT'S OWN NAME at a named
# non-PNC institution. v8 built this as a dd and it died at 8.4% coverage.
# Read as a dollar LEVEL and SHARE, which is the right scale for a sparse
# high-value event.
if RUN_DEST and exists(v8("selfpay")):
    SP = (spark.read.option("basePath", v8("selfpay")).parquet(v8("selfpay"))
          .join(YMMAP, "ym", "inner"))
    tot_out = (spark.read.parquet(v2("panel_pay_features"))
               .select("cust_pwr_id", "m_idx", F.coalesce("amt_out", F.lit(0.0)).alias("amt_out")))
    SPM = (SP.groupBy("cust_pwr_id", "m_idx")
           .agg(F.sum("amt").alias("self_amt"), F.countDistinct("fkey").alias("self_fin_n"),
                F.sum("n_txn").alias("self_txn"))
           .join(tot_out, ["cust_pwr_id", "m_idx"], "left")
           .withColumn("self_share", F.when(F.col("amt_out") > 0,
                                            F.col("self_amt")/F.col("amt_out"))))
    SPM.write.mode("overwrite").partitionBy("m_idx").parquet(hp("selfpay_level"))
    SPM = spark.read.parquet(hp("selfpay_level")).persist(StorageLevel.DISK_ONLY)

    CURVE = (SPM.join(att, "cust_pwr_id", "left")
             .withColumn("cohort", F.when(F.col("event_m").isNotNull(), "attriter")
                                    .otherwise("stayer"))
             .withColumn("rel_m", F.when(F.col("event_m").isNotNull(),
                                         F.col("m_idx")-F.col("event_m"))
                                   .otherwise(F.lit(None)))
             .filter((F.col("cohort") == "stayer") | F.col("rel_m").between(-12, 0)))
    A = (CURVE.filter(F.col("cohort") == "attriter").groupBy("rel_m").agg(
            F.count(F.lit(1)).alias("client_months"),
            F.sum("self_amt").alias("self_dollars"),
            F.expr("percentile_approx(self_share, 0.5)").alias("median_share"),
            F.avg("self_share").alias("mean_share"),
            F.avg("self_fin_n").alias("mean_destinations")).orderBy("rel_m")).toPandas()
    base = (CURVE.filter(F.col("cohort") == "stayer")
            .agg(F.avg("self_share").alias("mean_share"),
                 F.expr("percentile_approx(self_share, 0.5)").alias("median_share"),
                 F.sum("self_amt").alias("self_dollars")).collect()[0])
    A["stayer_mean_share"] = float(base["mean_share"] or 0)
    A["lift_vs_stayer"] = (A.mean_share/A.stayer_mean_share).round(3)
    disp(usd_col(A, ["self_dollars"]),
         title="5c1 &middot; <b>Money the client sent to its own name at another bank</b>, by "
               "month before exit, against the stayer baseline. This is the one destination "
               "measure that is unambiguous", n=20, save="v9_selfpay_curve")

    TOPSELF = (SPM.join(att, "cust_pwr_id", "inner")
               .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
               .filter(F.col("rel_m").between(*DEST_WINDOW)))
    TS = (spark.read.option("basePath", v8("selfpay")).parquet(v8("selfpay"))
          .join(YMMAP, "ym", "inner").join(att, "cust_pwr_id", "inner")
          .withColumn("rel_m", F.col("m_idx")-F.col("event_m"))
          .filter(F.col("rel_m").between(*DEST_WINDOW))
          .groupBy("fkey").agg(F.sum("amt").alias("dollars"),
                               F.countDistinct("cust_pwr_id").alias("clients"))
          .orderBy(F.desc("dollars")).limit(30))
    disp(usd_col(collect_pd(TS, "self-pay destinations"), "dollars"),
         title="5c2 &middot; <b>Named institutions receiving self-directed transfers from "
               "departing clients.</b> This is the closest thing in the data to a competitor "
               "list — and it needs a compliance read before it leaves the team",
         n=30, save="v9_selfpay_destinations")
    note("MONEY", "Where does the money go?",
         "see v9_destinations_* and v9_selfpay_*",
         "Gross destinations are confounded by supplier banking. Self-directed transfers are the "
         "only unambiguous measure. Both are bounded by the reconciliation in 5a.")


## 6 · Balance as a feature — it never has been

`bal_live` enters the model only as a **dd**: this month against the client's own trailing window,
divided by the peer median. That deliberately removes the *level*. And the peer decile is frozen
from the client's first three months, so **current size is nowhere in the model**.

Four cheap additions, none of which have been tested:

| Feature | Why |
|---|---|
| `log_bal` | The level itself. Large and small clients may simply churn differently |
| `bal_cv12` | Balance volatility. A client whose balance swings has a different risk profile from a steady one |
| `bal_share_top_acct` | Concentration of the balance in one account — a proxy for how easy it is to move |
| `bal_vs_peak` | Current balance against its own 12-month peak — a drawdown measure that the dd does not capture |

Tested as one block against the v8b shipping specification.

In [ ]:
# =====================================================================
# 6 · BALANCE-LEVEL FEATURE BLOCK                        [OUTPUT BLOCK 6]
# =====================================================================
if RUN_FEATURES:
    t0 = time.time()
    acct = spark.read.parquet(v2("panel_account_month")).filter(F.col("ym") >= DATE_START[:7])
    # the account panel's balance column has been named differently across
    # versions — resolve it rather than guessing, and degrade gracefully
    bcol = next((c for c in ("bal_live", "balance", "bal", "bal_eom") if c in acct.columns), None)
    print(f"  account balance column resolved to: {bcol}")
    TA = (acct.filter(F.col(bcol) > 0).groupBy("cust_pwr_id", "m_idx")
          .agg(F.max(bcol).alias("top_acct_bal"), F.sum(bcol).alias("sum_acct_bal"))
          .withColumn("bal_share_top_acct",
                      F.when(F.col("sum_acct_bal") > 0,
                             F.col("top_acct_bal")/F.col("sum_acct_bal"))))
    BALF = (BAL.select("cust_pwr_id", "m_idx", "bal_now", "bal_med12", "bal_peak12", "bal_cv12")
            .withColumn("log_bal", F.log1p(F.greatest(F.col("bal_now"), F.lit(0.0))))
            .withColumn("bal_vs_peak", F.when(F.col("bal_peak12") > BAL_FLOOR,
                                              F.col("bal_now")/F.col("bal_peak12")))
            .join(TA.select("cust_pwr_id", "m_idx", "bal_share_top_acct"),
                  ["cust_pwr_id", "m_idx"], "left") if bcol else BAL.select(
                "cust_pwr_id", "m_idx", "bal_now", "bal_med12", "bal_peak12", "bal_cv12")
            .withColumn("log_bal", F.log1p(F.greatest(F.col("bal_now"), F.lit(0.0))))
            .withColumn("bal_vs_peak", F.when(F.col("bal_peak12") > BAL_FLOOR,
                                              F.col("bal_now")/F.col("bal_peak12")))
            .withColumn("bal_share_top_acct", F.lit(None).cast("double"))
            .select("cust_pwr_id", "m_idx", "log_bal", "bal_cv12", "bal_vs_peak",
                    "bal_share_top_acct"))
    # These are LEVELS, not dd. They enter raw (standardised in the fit) with a
    # missingness indicator, because a level has no peer-relative meaning and
    # forcing it through the dd transform is what killed the ratio features in v8.
    sel = [F.col("cust_pwr_id"), F.col("m_idx")]
    for c in ["log_bal", "bal_cv12", "bal_vs_peak", "bal_share_top_acct"]:
        sel.append(F.coalesce(F.col(c), F.lit(0.0)).alias(f"ld_bl_{c}"))
        sel.append(F.when(F.col(c).isNull(), F.lit(1.0)).otherwise(F.lit(0.0)).alias(f"md_bl_{c}"))
    BALF = BALF.select(*sel)
    BALF.write.mode("overwrite").partitionBy("m_idx").parquet(hp("balance_features"))
    BALF = spark.read.parquet(hp("balance_features")).persist(StorageLevel.DISK_ONLY)

    RS = spark.read.parquet(v8("risk_set_v8b"))
    RISK9 = (RS.join(BALF, ["cust_pwr_id", "m_idx"], "left")
             .join(BAL.select("cust_pwr_id", "m_idx",
                              F.col("bal_now").alias("bar_now"),
                              F.col("bal_med12").alias("bar_med12"),
                              F.col("bal_peak12").alias("bar_peak12")),
                   ["cust_pwr_id", "m_idx"], "left"))
    RISK9.write.mode("overwrite").partitionBy("m_idx").parquet(hp("risk_set_v9"))
    RISK9 = spark.read.parquet(hp("risk_set_v9")).persist(StorageLevel.DISK_ONLY)
    kv([("risk-set columns, v8b", len(RS.columns)),
        ("risk-set columns, v9", len(RISK9.columns)),
        ("balance-level features added", 4),
        ("balance-at-risk columns carried", 3),
        ("block 6 wall (s)", round(time.time()-t0))],
       title="6a &middot; The merged panel", save="v9_panel_shape")


## 7 · Scoring, calibration, and the check that has to pass first

Everything from here multiplies a probability by dollars, so **the probability has to be on the
right scale, not just in the right order**. Two checks run before the value queue is built, and if
either fails, §8 is void:

1. **Calibration.** Predicted probability against realised rate, by predicted decile. The King &
   Zeng prior correction should put it near the diagonal.
2. **Does the model work on large clients?** Precision and AUC by balance decile. A value-weighted
   queue built on a model that is weaker at the top is worthless.

The specification is the v8b winner, `+ fin_in + cpty_acct_out`, plus the balance-level block from
§6 if it earns its place.

In [ ]:
# =====================================================================
# 7a · FIT, SCORE, CALIBRATE                             [OUTPUT BLOCK 7]
# =====================================================================
ORIGINS = list(range(M_MIN + ORIGIN_START_OFF, M_MAX - PRIMARY_H + 1))
assert 1 <= len(ORIGINS) <= MAX_ORIGINS, (
    f"{len(ORIGINS)} origins — m_idx is ABSOLUTE ({M_MIN}-{M_MAX}) and "
    f"ORIGIN_START_OFF is an OFFSET")
print(f"  origins m_idx {ORIGINS[0]}-{ORIGINS[-1]} ({len(ORIGINS)} folds)")

ALLC = RISK9.columns
NEWP = ("cptyn_", "cptya_", "fin_out2", "fin_in", "tim_", "railmix", "conc_",
        "selfpay", "acc_", "bl_")
V7_LD = sorted([c for c in ALLC if c.startswith("ld_")
                and not any(c.startswith("ld_"+p) for p in NEWP)])
V7_MD = [f"md_{c[3:]}" for c in V7_LD if f"md_{c[3:]}" in ALLC]
def blk(*pfx):
    ld = sorted([c for c in ALLC if any(c.startswith("ld_"+p) for p in pfx)])
    return ld + [f"md_{c[3:]}" for c in ld if f"md_{c[3:]}" in ALLC]
B_FIN, B_CPT, B_BAL = blk("fin_in"), blk("cptya_out"), blk("bl_")
BASE = V7_LD + V7_MD
SPECS = {
    "v8b_winner": BASE + B_FIN + B_CPT,
    "v8b_winner + balance_level": BASE + B_FIN + B_CPT + B_BAL,
    "v7_baseline": BASE,
}
disp(pd.DataFrame([dict(spec=k, n_features=len(v)) for k, v in SPECS.items()]),
     title="7a &middot; Specifications carried into the money work", save="v9_specs")

NEED = sorted(set(["cust_pwr_id", "m_idx", "event_A", "event_B",
                   "bar_now", "bar_med12", "bar_peak12"] +
                  [c for v in SPECS.values() for c in v]) & set(ALLC))
is_pos = (F.col("event_A").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)) |
          F.col("event_B").between(F.col("m_idx")+1, F.col("m_idx")+max(HORIZONS)))
TR = collect_pd(RISK9.filter(F.col("m_idx") <= max(ORIGINS) - min(HORIZONS))
                .withColumn("_u", (F.abs(F.hash(F.concat_ws("|", "cust_pwr_id",
                            F.col("m_idx").cast("string"), F.lit(SEED)))) % 100000)/100000.0)
                .filter(is_pos | (F.col("_u") < NEG_SAMPLE)).select(*NEED), "TRAIN")
TE = {t: collect_pd(RISK9.filter(F.col("m_idx") == t).select(*NEED), f"TEST {t}")
      for t in ORIGINS}

def prep(d):
    d = d.copy()
    for c in d.columns:
        if c.startswith("ld_"):   d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0)
        elif c.startswith("md_"): d[c] = pd.to_numeric(d[c], errors="coerce").fillna(1.0)
    for c in ["bar_now", "bar_med12", "bar_peak12"]:
        d[c] = pd.to_numeric(d[c], errors="coerce").fillna(0.0).clip(lower=0)
    return d
def label(d, H=PRIMARY_H):
    ev = pd.to_numeric(d["event_A"], errors="coerce"); t = pd.to_numeric(d["m_idx"], errors="coerce")
    y = ((ev > t) & (ev <= t+H)).astype(float); obs = (t+H <= M_MAX) | (y == 1)
    o = d.loc[obs].copy(); o["y"] = y.loc[obs].values; o["event_m"] = ev.loc[obs].values
    return o
TRp = prep(TR); TEp = {k: prep(v) for k, v in TE.items()}

SC, folds = [], []
for Tm in ORIGINS:
    tr = label(TRp[TRp.m_idx <= Tm - PRIMARY_H]); te = label(TEp[Tm])
    if tr.y.sum() < MIN_TRAIN_POS or te.y.sum() < 1: continue
    out = te[["cust_pwr_id", "m_idx", "y", "event_m", "bar_now", "bar_med12", "bar_peak12"]].copy()
    for nm, cols in SPECS.items():
        cols = [c for c in cols if c in tr.columns]
        sp = fit_spec(tr, cols)
        out[f"p_{nm}"] = predict_p(sp, te)
        folds.append(dict(spec=nm, origin=Tm, auc=auc(te.y.values, out[f"p_{nm}"].values),
                          n_feat=len(sp["cols"]) if sp else 0, base=float(te.y.mean())))
    SC.append(out)
S = pd.concat(SC, ignore_index=True)
FOLD = pd.DataFrame(folds)
SUM = FOLD.groupby("spec", as_index=False).agg(auc=("auc", "mean"), sd=("auc", "std"),
                                               n_feat=("n_feat", "max"))
disp(SUM.round(4).sort_values("auc", ascending=False),
     title="7b &middot; Does the balance-level block earn its place? "
           "<b>Ship threshold is the same as v8b — it has to move precision, not just AUC</b>",
     save="v9_balance_block")
PRIMARY = "v8b_winner + balance_level" if (
    float(SUM.loc[SUM.spec == "v8b_winner + balance_level", "auc"].iloc[0]) >
    float(SUM.loc[SUM.spec == "v8b_winner", "auc"].iloc[0]) + 0.002) else "v8b_winner"
S["p"] = S[f"p_{PRIMARY}"]
print(f"  primary specification: {PRIMARY}")

# ── calibration ───────────────────────────────────────────────────────
S["p_decile"] = pd.qcut(S.p.rank(method="first"), 10, labels=False) + 1
CAL = (S.groupby("p_decile", as_index=False)
       .agg(n=("y", "size"), predicted=("p", "mean"), realised=("y", "mean"),
            dollars=("bar_med12", "sum")))
CAL["ratio"] = (CAL.realised/CAL.predicted).round(3)
disp(usd_col(CAL.round(5), "dollars"),
     title="7c &middot; <b>Calibration.</b> Predicted against realised, by predicted decile. "
           "<code>ratio</code> near 1.0 means the probability is on the right scale — required "
           "before it is multiplied by dollars", save="v9_calibration")
_cal_err = float(np.abs(CAL.realised - CAL.predicted).sum()/CAL.realised.sum())
kv([("primary specification", PRIMARY),
    ("mean AUC across folds", round(float(SUM.loc[SUM.spec == PRIMARY, "auc"].iloc[0]), 4)),
    ("scored client-months", f"{len(S):,}"),
    ("overall predicted rate", f"{S.p.mean():.4%}"),
    ("overall realised rate", f"{S.y.mean():.4%}"),
    ("calibration error (sum |pred-real| / sum real)", round(_cal_err, 4)),
    ("verdict", "usable for expected-value ranking" if _cal_err < 0.25
                else "RECALIBRATE before trusting any dollar figure")],
   title="7d &middot; Calibration verdict", save="v9_calibration_verdict")
note("CALIB", "Is the probability on the right scale for expected-value ranking?",
     f"calibration error {_cal_err:.4f}",
     "The King & Zeng prior correction restores the population scale after case-control "
     "sampling. If this fails, every dollar figure in §8 is void.")


In [ ]:
# =====================================================================
# 9 · IS THE MODEL AS GOOD ON LARGE CLIENTS?             [OUTPUT BLOCK 8]
# =====================================================================
# Run BEFORE the value queue, because if it fails the value queue is void.
S["bal_decile"] = pd.qcut(S.bar_med12.rank(method="first"), 10, labels=False) + 1
rows = []
for d, g in S.groupby("bal_decile"):
    k = max(1, int(round(len(g)*0.003)))          # top 0.3% within the decile
    gg = g.sort_values("p", ascending=False)
    rows.append(dict(bal_decile=int(d), n=len(g),
                     median_balance=float(g.bar_med12.median()),
                     base_rate=float(g.y.mean()), auc=auc(g.y.values, g.p.values),
                     precision_top_0p3pct=float(gg.y.head(k).mean()),
                     lift=float(gg.y.head(k).mean()/g.y.mean()) if g.y.mean() > 0 else np.nan,
                     dollars_at_risk=float(g.loc[g.y == 1, "bar_med12"].sum())))
BD = pd.DataFrame(rows)
disp(usd_col(BD.round(4), ["median_balance", "dollars_at_risk"]),
     title="9a &middot; <b>Model quality by balance decile.</b> If AUC or lift falls off in "
           "deciles 9&ndash;10, a value-weighted queue is built on sand and §8 does not stand",
     n=12, save="v9_by_balance_decile")
_top = BD[BD.bal_decile >= 9].auc.mean(); _bot = BD[BD.bal_decile <= 2].auc.mean()
_all = float(SUM.loc[SUM.spec == PRIMARY, "auc"].iloc[0])
kv([("AUC, balance deciles 1-2", round(_bot, 4)),
    ("AUC, balance deciles 9-10", round(_top, 4)),
    ("AUC, whole book", round(_all, 4)),
    ("dollars at risk in deciles 9-10", usd(BD[BD.bal_decile >= 9].dollars_at_risk.sum())),
    ("share of all at-risk dollars there",
     f"{pct(BD[BD.bal_decile>=9].dollars_at_risk.sum(), BD.dollars_at_risk.sum()):.1%}"),
    ("verdict", "value ranking is defensible" if _top >= _all - 0.03
                else "MODEL IS WEAKER ON LARGE CLIENTS — fix before §8 is briefed")],
   title="9b &middot; The check that gates everything downstream", save="v9_decile_verdict")
note("BIGCLIENTS", "Is the model as good on the clients that hold the money?",
     f"AUC {_top:.4f} in deciles 9-10 vs {_all:.4f} overall",
     "A value-weighted queue concentrates on large clients. If discrimination is worse there, "
     "the expected-dollar ranking amplifies the model's weakest region.")


## 8 · The value-weighted queue and the savings model

### The ranking
```
score(c, t) = p(c, t) × bar(c, t)^α
```
`α = 0` is today's queue — pure probability, every client equal. `α = 1` is expected dollars at
risk. Values between temper the pull toward a handful of very large names. All are measured on the
same rows.

### The savings arithmetic
For a client caught by the queue, at the **first** month it appears (re-flags inside three months
are the same conversation):

```
saved(c) = 1{c exits within H} × bal_now(c, t_first) × p_save
```

**`bal_now`, not `bar_med12`** — an RM can only defend what is still on the books at the moment of
the call. The decay measured in §3 is therefore already inside this number rather than applied as
a separate haircut.

### What this is not
`p_save` has **no empirical support in this data**. Section 8 is a sensitivity grid over
plausible values, not a forecast. Four further caveats are printed with the output and belong on
any slide that uses these figures.

In [ ]:
# =====================================================================
# 8a · VALUE-WEIGHTED RANKING                            [OUTPUT BLOCK 9]
# =====================================================================
if RUN_QUEUE:
    S["bar"] = S["bar_med12"].where(S["bar_med12"] > BAL_FLOOR, 0.0)

    def first_alerts(df, score_col, K):
        """Top-K within each calendar month, then collapse to the FIRST month a
        client is alerted — a client in the list three months running is one
        conversation, not three."""
        d = df.copy()
        d["_r"] = d.groupby("m_idx")[score_col].rank(ascending=False, method="first",
                                                     na_option="bottom")
        fl = d[d._r <= K]
        # drop_duplicates keeps the WHOLE first row; groupby.first() skips NaN
        # per column and would silently mix values from different months
        first = fl.sort_values(["cust_pwr_id", "m_idx"]).drop_duplicates("cust_pwr_id",
                                                                        keep="first")
        return fl, first

    rows = []
    for a in VALUE_ALPHAS:
        S[f"s_a{a}"] = S.p * np.power(np.maximum(S.bar, 1.0), a)
        for K in CAPACITY:
            fl, first = first_alerts(S, f"s_a{a}", K)
            tp = first[first.y == 1]
            rows.append(dict(alpha=a, k=K, alerts=len(fl), conversations=len(first),
                             tp_clients=int(len(tp)),
                             precision_rows=float(fl.y.mean()),
                             precision_conversations=float(tp.shape[0]/max(len(first), 1)),
                             dollars_at_alert=float(tp.bar_now.sum()),
                             dollars_relationship=float(tp.bar_med12.sum()),
                             median_tp_balance=float(tp.bar_med12.median()) if len(tp) else np.nan,
                             median_lead=float((tp.event_m - tp.m_idx).median()) if len(tp) else np.nan))
    Q = pd.DataFrame(rows)
    Q.to_csv(OUT_DIR / "v9_value_queue.csv", index=False)

    disp(usd_col(Q[Q.k == QUEUE_K][["alpha", "k", "conversations", "tp_clients",
                                    "precision_conversations", "dollars_at_alert",
                                    "dollars_relationship", "median_tp_balance", "median_lead"]]
                 .round(4),
                 ["dollars_at_alert", "dollars_relationship", "median_tp_balance"]),
         title=f"8a &middot; <b>The trade, at K={QUEUE_K}.</b> As &alpha; rises the queue catches "
               "<i>fewer clients</i> and <i>more money</i>. Read the two dollar columns together: "
               "one is what is still there to defend, the other is the size of the relationship",
         save="v9_alpha_tradeoff")

    piv = Q.pivot_table(index="k", columns="alpha", values="dollars_at_alert")
    disp(piv.apply(lambda c: c.map(usd)).reset_index(),
         title="8b &middot; <b>Defendable dollars reached, by list length and &alpha;.</b> "
               "&alpha;=0 is the queue we have today", save="v9_dollars_grid")
    piv2 = Q.pivot_table(index="k", columns="alpha", values="tp_clients")
    disp(piv2.reset_index(),
         title="8c &middot; The same grid in <b>clients caught</b> — the cost of the money view",
         save="v9_clients_grid")

    a0 = Q[(Q.alpha == 0.0) & (Q.k == QUEUE_K)].iloc[0]
    best = Q[Q.k == QUEUE_K].sort_values("dollars_at_alert").iloc[-1]
    kv([("today's queue (alpha=0), clients caught", int(a0.tp_clients)),
        ("today's queue, defendable dollars reached", usd(a0.dollars_at_alert)),
        ("best alpha at this capacity", best.alpha),
        ("...clients caught", int(best.tp_clients)),
        ("...defendable dollars reached", usd(best.dollars_at_alert)),
        ("extra dollars put in front of an RM", usd(best.dollars_at_alert - a0.dollars_at_alert)),
        ("multiple", round(best.dollars_at_alert/max(a0.dollars_at_alert, 1), 2)),
        ("clients given up to get there", int(a0.tp_clients - best.tp_clients))],
       title=f"8d &middot; <b>The headline trade at {QUEUE_K} alerts a month</b>",
       save="v9_alpha_headline")
    note("VALUEQ", "Does ranking on expected dollars beat ranking on probability?",
         f"alpha={best.alpha} reaches {usd(best.dollars_at_alert)} vs {usd(a0.dollars_at_alert)}",
         "Same capacity, same clients, same folds. The cost is clients caught; the gain is money "
         "reached. Which one the business wants is a decision, not a modelling result.")


In [ ]:
# =====================================================================
# 8e · THE SAVINGS MODEL — capacity x RM success rate     [OUTPUT BLOCK 10]
# =====================================================================
if RUN_SAVINGS:
    ALPHA_STAR = float(best.alpha)
    rows = []
    for K in CAPACITY:
        for a in sorted({0.0, ALPHA_STAR}):
            fl, first = first_alerts(S, f"s_a{a}", K)
            tp = first[first.y == 1]
            reach = float(tp.bar_now.sum())
            for ps in P_SAVE_GRID:
                saved = reach*ps
                conv = len(first)
                cost = conv*RM_COST_PER_CALL
                rows.append(dict(alpha=a, k=K, p_save=ps,
                                 conversations=conv, tp_clients=int(len(tp)),
                                 dollars_reached=reach, dollars_saved=saved,
                                 dollars_saved_annualised=saved*ANNUALISE,
                                 rm_cost=cost, net=saved-cost,
                                 saved_per_conversation=saved/max(conv, 1),
                                 cost_per_dollar_saved=cost/max(saved, 1),
                                 breakeven_p_save=cost/max(reach, 1)))
    SAV = pd.DataFrame(rows)
    SAV.to_csv(OUT_DIR / "v9_savings.csv", index=False)

    G = SAV[SAV.alpha == ALPHA_STAR].pivot_table(index="k", columns="p_save",
                                                 values="dollars_saved_annualised")
    disp(G.apply(lambda c: c.map(usd)).reset_index(),
         title=f"8e &middot; <b>Annualised dollars retained</b> — alert volume (rows) &times; RM "
               f"save rate (columns), at &alpha;={ALPHA_STAR:g}. <b>The columns are an "
               f"assumption with no data behind it</b>", save="v9_savings_grid")

    G0 = SAV[SAV.alpha == 0.0].pivot_table(index="k", columns="p_save",
                                           values="dollars_saved_annualised")
    DIFF = (G - G0)
    disp(DIFF.apply(lambda c: c.map(usd)).reset_index(),
         title="8f &middot; <b>What the value ranking adds</b> over today's probability ranking, "
               "same capacity and same save rate", save="v9_savings_uplift")

    BE = (SAV[(SAV.alpha == ALPHA_STAR) & (SAV.p_save == P_SAVE_BASE)]
          [["k", "conversations", "tp_clients", "dollars_reached", "dollars_saved",
            "dollars_saved_annualised", "rm_cost", "net", "saved_per_conversation",
            "breakeven_p_save"]])
    disp(usd_col(BE.round(4), ["dollars_reached", "dollars_saved", "dollars_saved_annualised",
                               "rm_cost", "net", "saved_per_conversation"]),
         title=f"8g &middot; <b>The business case at a {P_SAVE_BASE:.0%} save rate</b> and "
               f"{usd(RM_COST_PER_CALL)} per RM outreach. <code>breakeven_p_save</code> is the "
               f"success rate at which the calls pay for themselves", save="v9_business_case")

    _b = BE[BE.k == 1000].iloc[0] if 1000 in set(BE.k) else BE.iloc[len(BE)//2]
    kv([("alert volume", f"{int(_b.k):,} a month"),
        ("distinct conversations over 7 months", f"{int(_b.conversations):,}"),
        ("departing clients reached", f"{int(_b.tp_clients):,}"),
        ("defendable dollars put in front of an RM", usd(_b.dollars_reached)),
        (f"retained at a {P_SAVE_BASE:.0%} save rate, annualised",
         usd(_b.dollars_saved_annualised)),
        ("RM cost of those conversations", usd(_b.rm_cost)),
        ("net", usd(_b.net)),
        ("dollars retained per conversation", usd(_b.saved_per_conversation)),
        ("break-even save rate", f"{_b.breakeven_p_save:.3%}")],
       title="8h &middot; <b>One row, spelled out</b>", save="v9_case_headline")

    disp(pd.DataFrame([
        dict(caveat="RM save rate has no empirical support in this data",
             effect="Every figure scales linearly with it. Only a pilot settles it"),
        dict(caveat="'Saved' assumes the balance is retained in full",
             effect="Optimistic. A partial retention or a delayed exit both count as a save here"),
        dict(caveat="Some alerted clients would have stayed anyway",
             effect="The counterfactual is unobserved. A holdout arm in the pilot is the only fix"),
        dict(caveat="bal_now at the alert month is used, not the relationship size",
             effect="Conservative — the decay in §3 is already inside the number"),
        dict(caveat="B_bal_exit clients are not counted as events here",
             effect="Understates the pool. They drain without closing and their money leaves too"),
        dict(caveat="RM cost is a placeholder",
             effect=f"Set at {usd(RM_COST_PER_CALL)} per outreach. Override RM_COST_PER_CALL"),
    ]), title="8i &middot; <b>Print this next to any savings figure</b>", save="v9_caveats")
    note("SAVINGS", "What could a queue plausibly retain?",
         f"{usd(_b.dollars_saved_annualised)} annualised at {int(_b.k)} alerts/month and a "
         f"{P_SAVE_BASE:.0%} save rate",
         "Sensitivity grid, not a forecast. Break-even save rate is the number to argue about — "
         "it is derived, not assumed.")


---

## After this run

1. **§9b gates everything.** If AUC in balance deciles 9–10 is materially below the whole-book
   figure, stop and fix that before any dollar number leaves the team. A value-weighted queue
   concentrates precisely on the region where the model would then be weakest.

2. **§7c gates the dollar arithmetic.** Expected value needs a calibrated probability, not just a
   correct ordering. If the calibration ratio drifts from 1.0 in the top deciles, apply an isotonic
   or Platt recalibration on the training folds before §8 is quoted.

3. **Argue about the break-even save rate, not the savings figure.** `breakeven_p_save` in 8g is
   *derived* from reach and cost. If it comes out at 2%, the programme is defensible under almost
   any assumption. If it comes out at 25%, the case rests entirely on an unmeasured quantity and
   the pilot is the only way forward.

4. **§5a bounds §5b and §5c.** If most of the balance decline is not visible as payment flow, the
   destination analysis describes a minority of the money and must be presented that way.

5. **Do not circulate 5c2 without a compliance read.** A list of named institutions receiving
   self-directed transfers from departing clients is competitive intelligence derived from customer
   payment data. The permissible-use question is the same one already flagged for counterparty
   prospecting.

6. **The obvious next measurement.** Re-run §3 and §8 on `B_bal_exit`. Those clients drain without
   closing, they are invisible to balance-based monitoring, and **their money leaves too** — so
   the addressable pool in §2 is understated by however much they hold. That is likely the single
   largest missing number in the business case.
